In [170]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [171]:
df = pd.read_excel('Data_Mahasiswa_UNMUH.xlsx')

df.head()

,Nama,Jenis Kelamin,Program Studi,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,...,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi,Estimasi Masa Studi
0,Sinta,Perempuan,PGSD,2020,2024,3.70,3.62,3.66,3.75,3.59,...,NaN,151,0,rendah,sedang,sedang,menengah,bekerja,tidak aktif,4.6
1,Sasi Mardita,Perempuan,PGSD,2021,2025,3.78,3.62,3.65,3.84,3.90,...,NaN,151,0,sangat tinggi,sangat tinggi,rendah,menengah,tidak bekerja,aktif,4.0
2,Muhammad Farhan Sugiwangsa,Laki-laki,Teknik Sipil,2021,2025,3.60,3.60,3.65,3.70,3.68,...,NaN,144,0,sedang,sedang,sedang,menengah,tidak bekerja,tidak aktif,3.7
3,Lilis Karsina,Perempuan,KSDA,2020,2025,3.48,3.30,3.40,3.35,3.45,...,NaN,145,0,rendah,sedang,sedang,menengah,tidak bekerja,tidak aktif,4.3
4,Muhammad Miraldy,Laki-laki,Teknik Sipil,2021,2025,3.39,3.25,3.30,3.30,3.35,...,NaN,146,0,rendah,tinggi,sedang,tinggi,tidak bekerja,tidak aktif,3.8


In [172]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 278 entries, 0 to 277
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Nama                             278 non-null    str    
 1   Jenis Kelamin                    278 non-null    str    
 2   Program Studi                    278 non-null    str    
 3   Tahun Masuk                      278 non-null    int64  
 4   Tahun Lulus                      278 non-null    int64  
 5   IPK                              278 non-null    float64
 6   IPS 1                            278 non-null    float64
 7   IPS 2                            278 non-null    float64
 8   IPS 3                            278 non-null    float64
 9   IPS 4                            278 non-null    float64
 10  IPS 5                            278 non-null    float64
 11  IPS 6                            278 non-null    float64
 12  IPS 7                            

Nama                                 0
Jenis Kelamin                        0
Program Studi                        0
Tahun Masuk                          0
Tahun Lulus                          0
IPK                                  0
IPS 1                                0
IPS 2                                0
IPS 3                                0
IPS 4                                0
IPS 5                                0
IPS 6                                0
IPS 7                                0
IPS 8                              122
IPS 9                              182
IPS 10                             267
IPS 11                             264
Jumlah SKS                           0
Jumlah Mata Kuliah yang Diulang      0
Motivasi Belajar                     0
Dukungan Keluarga                    0
Tingkat Stres                        0
Sosial-Ekonomi                       0
Pekerjaan Paruh Waktu                0
Keaktifan dalam Berorganisasi        0
Estimasi Masa Studi      

In [173]:
# mapping manual
mapping_5 = {
    'sangat rendah':1,'rendah':2,'sedang':3,'tinggi':4,'sangat tinggi':5
}

mapping_stres = {'rendah':1,'sedang':2,'tinggi':3}
mapping_ekonomi = {'rendah':1,'menengah':2,'tinggi':3}
mapping_kerja = {'tidak bekerja':0,'bekerja':1}
mapping_org = {'tidak aktif':0,'aktif':1}

# kolom kategori
cols_kategori = [
    'Motivasi Belajar',
    'Dukungan Keluarga',
    'Tingkat Stres',
    'Sosial-Ekonomi',
    'Pekerjaan Paruh Waktu',
    'Keaktifan dalam Berorganisasi'
]

# ubah ke huruf kecil
for col in cols_kategori:
    df[col] = df[col].astype(str).str.lower().str.strip()

# mapping
df['Motivasi Belajar'] = df['Motivasi Belajar'].map(mapping_5)
df['Dukungan Keluarga'] = df['Dukungan Keluarga'].map(mapping_5)
df['Tingkat Stres'] = df['Tingkat Stres'].map(mapping_stres)
df['Sosial-Ekonomi'] = df['Sosial-Ekonomi'].map(mapping_ekonomi)
df['Pekerjaan Paruh Waktu'] = df['Pekerjaan Paruh Waktu'].map(mapping_kerja)
df['Keaktifan dalam Berorganisasi'] = df['Keaktifan dalam Berorganisasi'].map(mapping_org)


In [174]:
ips_cols = [f'IPS {i}' for i in range(1, 12)]

# pastikan semua IPS ada
for col in ips_cols:
    if col not in df.columns:
        df[col] = np.nan

# isi yang kosong dengan rata-rata
df[ips_cols] = df[ips_cols].apply(
    lambda row: row.fillna(row.mean()), axis=1
)


In [175]:
df['IPK'] = df['IPK'].clip(0, 4)
df['Jumlah Mata Kuliah yang Diulang'] = df['Jumlah Mata Kuliah yang Diulang'].clip(0, 10)

In [176]:
df = df.drop([
    'Nama',
    'Program Studi',
    'Jenis Kelamin',
], axis=1, errors='ignore')

In [177]:
y = df['Estimasi Masa Studi']
X = df.drop('Estimasi Masa Studi', axis=1)

In [178]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [179]:
X_train

,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,IPS 5,IPS 6,IPS 7,...,IPS 10,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi
260,2021,2025,3.79,3.75,3.78,3.80,3.82,3.79,3.81,3.77,...,3.78875,3.78875,144,0,2,3,1,3,0,0
124,2020,2024,3.60,3.45,3.50,3.55,3.60,3.68,3.65,3.70,...,3.59500,3.59500,148,0,5,5,3,3,0,0
33,2021,2025,3.91,3.95,3.90,3.80,3.85,3.92,3.98,3.94,...,3.91000,3.91000,140,0,4,4,1,2,0,1
86,2020,2024,3.81,3.68,3.63,3.71,3.83,3.88,3.86,3.84,...,3.79125,3.79125,151,0,3,4,2,2,1,0
264,2021,2025,3.49,3.40,3.45,3.48,3.50,3.47,3.51,3.49,...,3.47750,3.47750,145,0,3,3,2,2,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,2021,2025,3.80,3.69,3.75,3.88,3.82,3.85,3.82,3.79,...,3.80375,3.80375,151,0,3,3,1,3,0,0
71,2020,2024,3.82,3.60,3.70,3.78,3.84,3.90,3.95,3.91,...,3.81875,3.81875,151,0,4,4,2,2,0,0
106,2020,2024,3.79,3.62,3.71,3.76,3.80,3.85,3.89,3.87,...,3.79625,3.79625,151,0,3,3,2,2,0,0
270,2021,2025,3.81,3.77,3.79,3.81,3.82,3.84,3.80,3.86,...,3.81000,3.81000,146,0,3,3,2,2,0,0


In [180]:
model = RandomForestRegressor(
    n_estimators=700,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",700
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [181]:
X_test

,Tahun Masuk,Tahun Lulus,IPK,IPS 1,IPS 2,IPS 3,IPS 4,IPS 5,IPS 6,IPS 7,...,IPS 10,IPS 11,Jumlah SKS,Jumlah Mata Kuliah yang Diulang,Motivasi Belajar,Dukungan Keluarga,Tingkat Stres,Sosial-Ekonomi,Pekerjaan Paruh Waktu,Keaktifan dalam Berorganisasi
30,2020,2024,3.40,3.30,3.30,3.20,3.35,3.40,3.40,3.55,...,3.38125,3.38125,152,0,3,3,2,1,0,1
126,2021,2025,3.93,3.90,3.92,3.95,4.00,3.98,3.94,3.90,...,3.93000,3.93000,146,0,5,5,2,3,0,0
199,2021,2025,3.79,3.72,3.78,3.85,3.88,3.83,3.80,3.77,...,3.79625,3.79625,151,0,3,3,1,3,0,0
142,2021,2025,3.83,3.75,3.80,3.86,3.88,3.85,3.83,3.81,...,3.83500,3.83500,151,0,3,4,1,2,0,1
253,2021,2025,3.84,3.82,3.80,3.88,3.85,3.83,3.84,3.86,...,3.84000,3.84000,146,0,3,3,1,3,0,0
237,2020,2025,3.79,3.65,3.72,3.76,3.83,3.80,3.88,3.93,...,3.75000,3.79000,151,0,2,3,1,3,0,0
97,2020,2024,3.76,3.66,3.71,3.76,3.81,3.85,3.83,3.79,...,3.76000,3.76000,151,0,2,1,1,2,0,0
206,2021,2025,3.87,3.78,3.84,3.89,3.93,3.96,3.91,3.88,...,3.87000,3.87000,151,0,5,4,1,2,0,0
263,2021,2025,3.61,3.55,3.58,3.60,3.63,3.59,3.62,3.61,...,3.60250,3.60250,144,0,3,3,2,2,0,1
144,2021,2025,3.83,3.76,3.82,3.88,4.00,3.78,3.84,3.80,...,3.83875,3.83875,151,0,3,4,1,2,0,0


In [182]:
y_pred = model.predict(X_test)

In [183]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nEvaluasi Model:")
print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)


Evaluasi Model:
MAE: 0.15415023797626576
MSE: 0.038334386702301305
R2 Score: 0.7427695801895433


In [184]:
joblib.dump(model, 'model_random_forest.pkl')

['model_random_forest.pkl']

In [185]:
joblib.dump(X.columns.tolist(), "fitur_model.pkl")

print("\n✅ Model & fitur berhasil disimpan!")


✅ Model & fitur berhasil disimpan!


In [186]:
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the trained model
loaded_model = joblib.load('model_random_forest.pkl')

# Load the feature names used during training
loaded_features = joblib.load('fitur_model.pkl')

# Ensure X_test columns match the loaded_features order
X_test_aligned = X_test[loaded_features]
# Make predictions using the loaded model
loaded_model_predictions = loaded_model.predict(X_test_aligned)

# Display some predictions
print("\n--- Hasil Prediksi dari Model yang Dimuat ---")
print("Prediksi: ", loaded_model_predictions[:5])
print("Aktual: ", y_test.head().values)

# Evaluate the loaded model
mae_loaded = mean_absolute_error(y_test, loaded_model_predictions)
mse_loaded = mean_squared_error(y_test, loaded_model_predictions)
r2_loaded = r2_score(y_test, loaded_model_predictions)

print("\n--- Evaluasi Model yang Dimuat ---")
print(f"MAE (Loaded Model): {mae_loaded}")
print(f"MSE (Loaded Model): {mse_loaded}")
print(f"R2 Score (Loaded Model): {r2_loaded}")


--- Hasil Prediksi dari Model yang Dimuat ---
Prediksi:  [4.19541802 3.69814286 4.15385261 3.94019355 3.98921985]
Aktual:  [4.1 3.9 4.2 4.  4.4]

--- Evaluasi Model yang Dimuat ---
MAE (Loaded Model): 0.15415023797626576
MSE (Loaded Model): 0.038334386702301305
R2 Score (Loaded Model): 0.7427695801895433
